## Data Ingestion

In [1]:
import langchain
print(langchain.__version__)

1.3.1


In [2]:
from langchain_core.documents import Document

In [3]:
## PdfLoaders
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, DirectoryLoader

## load all the pdf files from the dir.
Dir_loaders = DirectoryLoader(
    "../data/pdf",
    glob="python-crash-course.pdf",
    # glob = "**/*.pdf",
    loader_cls= PyMuPDFLoader,
    show_progress = False 
)
pdf_documents = Dir_loaders.load();

/workspaces/Python_RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data chunking

In [4]:
# Text Chunking for RAG Pipeline

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Function to split documents into chunks
def create_chunk(documents, chunk_size=1000, chunk_overlap=200):

    """
    documents       -> List of LangChain Document objects
    chunk_size      -> Maximum characters per chunk
    chunk_overlap   -> Repeated characters between chunks                                                                                               
    """

    # Create text splitter object
    text_splitter = RecursiveCharacterTextSplitter(

        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,

        # Split priority
        separators=[
            "\n\n",   # Paragraph split
            "\n",     # Line split
            " ",      # Word split
            ""        # Character split
        ]
    )

    # Split documents into smaller chunks
    split_docs = text_splitter.split_documents(documents)

    # Print information
    print(f"Total Documents Loaded : {len(documents)}")
    print(f"Total Chunks Created   : {len(split_docs)}")

    # Show sample chunk
    if split_docs:

        print("\n================ SAMPLE CHUNK ================\n")

        print(split_docs[0].page_content[:500])

        print("\n================ METADATA ====================\n")

        print(split_docs[0].metadata)

    return split_docs


# ==========================================
# Create chunks from your loaded PDF files
# ==========================================

chunks = create_chunk(pdf_documents)


# ==========================================
# View first chunk completely
# ==========================================

print("\n================ FIRST CHUNK =================\n")

print(chunks[1].page_content)


# ==========================================
# View first chunk metadata
# ==========================================

print("\n================ CHUNK METADATA ==============\n")

print(chunks[1].metadata)


# ==========================================
# Total chunks created
# ==========================================

print(f"\nTotal Chunks : {len(chunks)}")
print(chunks[0].metadata)
print(chunks[10].metadata)
print(chunks[20].metadata)

Total Documents Loaded : 562
Total Chunks Created   : 1503

================ SAMPLE CHUNK ================

A  H A N D S - O N ,  P R O J E C T - B A S E D
I N T R O D U C T I O N  T O  P R O G R A M M I N G
E R I C  M A T T H E S
PY THON
CR ASH COURSE
PY THON
CR ASH COURSE

================ METADATA ====================

{'producer': 'Adobe PDF Library 9.9', 'creator': 'Adobe InDesign CS5.5 (7.5.3)', 'creationdate': '2015-10-26T15:01:49-07:00', 'source': '../data/pdf/python-crash-course.pdf', 'file_path': '../data/pdf/python-crash-course.pdf', 'total_pages': 562, 'format': 'PDF 1.6', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2015-10-27T15:56:06-07:00', 'trapped': '', 'modDate': "D:20151027155606-07'00'", 'creationDate': "D:20151026150149-07'00'", 'page': 0}

================ FIRST CHUNK =================

Python Crash Course

================ CHUNK METADATA ==============

{'producer': 'Adobe PDF Library 9.9', 'creator': 'Adobe InDesign CS5.5 (7.5.3)', 'cre

## Data Embedding 


In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer   
import chromadb
from chromadb import Settings
import uuid
from typing import List, Tuple, Dict, Any
from sklearn.metrics.pairwise import cosine_similarity
import os

In [6]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self.load_model()

    def load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Model Dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error occurred while loading the model: {e}")

    def create_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        try:
            print(f"Creating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print("Embeddings created successfully.")
            return embeddings
        except Exception as e:
            print(f"Error occurred while creating embeddings: {e}")
            return np.array([])
        


## Run the embedding manager
embedding_manager = EmbeddingManager()


Loading embedding model: all-MiniLM-L6-v2...
Model loaded successfully. Model Dimension: 384


## Vector store

In [7]:
class VectorStore:

    #Manages documents embeddings in chromadb vector database.

    def __init__(self, collection_name: str="pdf_documents", persist_directory: str="../data/vector_db"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self.initialize_store()


    def initialize_store(self):

        """Initialze the client and the collection."""
        try:
            # create a presistent chromaDB client 
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #get or create a collection to store the document chunks embeddings
            self.collection = self.client.get_or_create_collection(
                self.collection_name,
                metadata={"description":"collection of pdf document chunks"}
            )
            print(f"Vector store initialized successfully. Collection Name: {self.collection_name}")
            print(f"Existing documents in the collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error occurred while initializing vector store: {e}")   
            raise

    def add_document(self, documents: List[Any], embeddings: np.ndarray):
        
        """Add documents and their corresponding embeddings to the collection."""

        if len(documents) != len(embeddings):
            raise ValueError("The number of documents and embeddings must be the same.")
        
        print(f"Adding {len(documents)} documents to the vector store...")


        #prepare the data for chromaDB
        ids=[]
        metadatas=[]
        documents_content=[]
        embeddings_list=[]

        for i, (doc, embed) in enumerate(zip(documents, embeddings)): # Zip the documents and embeddings in to tuple
            
            # generate unique id
            doc_id = f"doc{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            #prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)
            
            #Document content
            documents_content.append(doc.page_content)

            #embeddings list
            embeddings_list.append(embed.tolist())

        try:
            self.collection.add(
                ids=ids,
                documents=documents_content,
                metadatas=metadatas,
                embeddings=embeddings_list
            )
            print(f"Documents added successfully. Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error occurred while adding documents to vector store: {e}")
            raise

        
vector_store = VectorStore()


Vector store initialized successfully. Collection Name: pdf_documents
Existing documents in the collection: 1503


In [8]:
chunks

[Document(metadata={'producer': 'Adobe PDF Library 9.9', 'creator': 'Adobe InDesign CS5.5 (7.5.3)', 'creationdate': '2015-10-26T15:01:49-07:00', 'source': '../data/pdf/python-crash-course.pdf', 'file_path': '../data/pdf/python-crash-course.pdf', 'total_pages': 562, 'format': 'PDF 1.6', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2015-10-27T15:56:06-07:00', 'trapped': '', 'modDate': "D:20151027155606-07'00'", 'creationDate': "D:20151026150149-07'00'", 'page': 0}, page_content='A  H A N D S - O N ,  P R O J E C T - B A S E D\nI N T R O D U C T I O N  T O  P R O G R A M M I N G\nE R I C  M A T T H E S\nPY THON\nCR ASH COURSE\nPY THON\nCR ASH COURSE'),
 Document(metadata={'producer': 'Adobe PDF Library 9.9', 'creator': 'Adobe InDesign CS5.5 (7.5.3)', 'creationdate': '2015-10-26T15:01:49-07:00', 'source': '../data/pdf/python-crash-course.pdf', 'file_path': '../data/pdf/python-crash-course.pdf', 'total_pages': 562, 'format': 'PDF 1.6', 'title': '', 'author': '', 'su

In [9]:
# #convert text to embeddings
# texts = [chunk.page_content for chunk in chunks]

# # generate embeddings
# embeddings = embedding_manager.create_embeddings(texts)

# # add to vector store
# vector_store.add_document(chunks, embeddings)    

## Retriever Pipeline form Vector Store.

In [10]:
class RAGRetriever:
    """ Handles a Query_based retrival fro the vector store."""

    def __init__ (self, vector_store: VectorStore, embedding_manager: EmbeddingManager):

        """ 
        vector_store: vector store containing Document embeddings
        embedding_manager: manager for generating query embeddings
        
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float=0.0 ) -> List[Dict[str, Any]]:

        """ Retrieve relavent documents from ve tore store"""

        """ Args:
                Query : The user Search Query.
                tpo_k : Number of top results to return.
                score_threshold : Minimum similarity score threshold
        """

        """Return : A List of Dict containing retrieved documents and metadata"""

        print(f"Retrieving documents for the Query: {query}")
        print(f"top K: {top_k}, score_threshold: {score_threshold}")

        # Create embedding sfor the query
        query_embedding = self.embedding_manager.create_embeddings([query])[0]

        # retrieve all documents and embeddings from the vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()], 
                n_results=top_k
            )

            #process the results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):

                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "Content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i+1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents with similarity score above the threshold from vector store.")
            else:
                print("No documents found in vector store.")
                
            return retrieved_docs
        
        except Exception as e:
            print(f"Error occurred during retrieval: {e}")
            return []   
            

rag_retriever = RAGRetriever(vector_store, embedding_manager) 
rag_retriever     


In [11]:
rag_retriever.retrieve("What is RAG?", top_k=3, score_threshold=0.5)

Retrieving documents for the Query: What is RAG?
top K: 3, score_threshold: 0.5
Creating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:02<00:00,  2.92s/it]

Embeddings created successfully.
Retrieved 0 documents with similarity score above the threshold from vector store.


[]

## Integration of VectorDB context Pipeline with LLM Output.

In [ ]:
# A simple RAG pipeline with Groq llm

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv   

load_dotenv()

# initilize the groq llm
groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(groq_api_key=groq_api_key,model="llama-3.1-8b-instant", temperature=0.1, max_tokens=1024)

# print(os.getenv("GROQ_API_KEY"))
# print(type(llm))

# simple RAG pipeline function
def rag_pipeline(query: str, retriever: RAGRetriever, llm: ChatGroq, topk_k=3):

    results = retriever.retrieve(query, top_k=topk_k)
    context ="\n\n".join([doc["Content"] for doc in results]) if results else ""
    if not context:
        return "No relevant documents found for the query."

    # generate answer using the Groq LLM
    prompt = f""" use the following context to answer the question consicely
                    Context: {context}
                    Question: {query}
                    Answer:"""

    response = llm.invoke(prompt.format(context=context, query=query))
    return response.content


In [18]:
#test the RAG pipeline
query = "what is a 'ship that fires bullets'?"
answer = rag_pipeline(query, rag_retriever, llm)
print(f"Query: {query}\n")
print(f"Answer: {answer}\n")

Retrieving documents for the Query: what is a 'ship that fires bullets'?
top K: 3, score_threshold: 0.0
Creating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 70.74it/s]

Embeddings created successfully.
Retrieved 1 documents with similarity score above the threshold from vector store.
Query: what is a 'ship that fires bullets'?

Answer: A 'ship that fires bullets' is a graphical representation of a spaceship in a game, capable of moving horizontally and firing projectiles (bullets) upwards at a specified speed.



In [23]:
## Advanced RAG pipeline with Groq LLM
def advanced_rag_pipeline(query: str, retriever: RAGRetriever, llm: ChatGroq, topk_k=3, min_score=0.0, return_context=False):

    results = retriever.retrieve(query, top_k=topk_k, score_threshold=min_score)
    if not results:
        return {'answer':"No relevant documents found for the query.", 'source':[], 'confidence':0.0, 'content':""}
    
    # prepare context for LLM
    context ="\n\n".join(doc['Content'] for doc in results)

    #prepare source information for llm
    sources = [{
        'source' : doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page' : doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview' : doc['Content'][:20],

    } for doc in results]
    
    confidence = max(doc['similarity_score'] for doc in results)

    # generate answer using the Groq LLM
    prompt = f""" Use the following context to answer the question concisely.
                Context: {context}
                Question: {query}
                Answer:"""
    
    response = llm.invoke(prompt.format(context=context, query=query))
    
    output ={
        'answer': response.content,
        'source': sources,
        'confidence': confidence,
        'content': context if return_context else ""
    }
    return output

In [27]:
result = advanced_rag_pipeline("what is a 'ship that fires bullets'?", rag_retriever, llm, topk_k=3, min_score=0.0, return_context=True)
print(f"Answer: {result['answer']}\n")
print(f"Sources: {result['source']}\n")
print(f"Confidence Score: {result['confidence']}\n")
print (f"Context: {result['content']}\n")

Retrieving documents for the Query: what is a 'ship that fires bullets'?
top K: 3, score_threshold: 0.0
Creating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 99.25it/s]

Embeddings created successfully.
Retrieved 1 documents with similarity score above the threshold from vector store.


Answer: A 'ship that fires bullets' is a game object in a space-themed video game, specifically a spaceship that can move left and right and shoot projectiles (bullets) upwards from its location.

Sources: [{'source': '../data/pdf/python-crash-course.pdf', 'page': 293, 'score': 0.09521830081939697, 'preview': 'If you run alien_inv'}]

Confidence Score: 0.09521830081939697

Context: If you run alien_invasion.py now, you should be able to move the ship 
right and left, and fire as many bullets as you want. The bullets travel up the 
screen and disappear when they reach the top, as shown in Figure 12-3. You 
can alter the size, color, and speed of the bullets in settings.py. 
Figure 12-3: The ship after firing a series of bullets
Deleting Old Bullets
At the moment, the bullets disappear when they reach the top, but only 
because Pygame can’t draw them above the top of the screen. The bullets 
actually continue to exist; their y-coordinate values just grow increasingly 
negative. This is a